# Deep Learning for Pendant-Drop Surface Tension Estimation

Project workflow:

1. Numerically solve the droplet shape equation.
2. Learn the relationship between contour features and surface tension using neural networks.
3. Apply the neural network to experimental contours to estimate liquid surface tension.

# 1. Numerical Solution

Refer to the underlying paper for the equation derivation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy.integrate import odeint
import pickle

In [ ]:
# Curve solver with the reference length R0 normalized to 1.
class YLCurve_beta():
    def __init__(self,beta=0.2,iterations=1000,srange=4.0,r0=1e-6,theta0=1e-6):
        #Initialize variables.
        self.beta=beta
        self.s=np.linspace(0,srange,iterations)
        self.curves=[]
        
        #Define the differential system for scipy.odeint.
        def YLE(y,s,beta):
            x,z,theta=y
            dx=np.cos(theta)
            dz=np.sin(theta)
            dtheta=2-beta*z-np.sin(theta)/x
            return np.array([dx,dz,dtheta])
        #y1: x (horizontal coordinate)
        #y2: z (vertical coordinate)
        #y3: theta (tangent angle)
        #(s is the arc length measured from the bottom tip.)
        # Nondimensionalize each length by the radius of curvature at the bottom tip.
        
        #Solve the Young-Laplace equation.
        z0=r0*theta0
        solution,self.information=odeint(YLE,[r0,z0,theta0],self.s,args=(self.beta,),full_output=1)
        self.solution=solution[solution[::,2]<np.pi]
        self.r=solution[::,0]
        self.z=solution[::,1]

    def curve(self):
        return np.vstack([self.r,self.z])

In [ ]:
# Example of solving a droplet curve.
# r and z are the horizontal and vertical coordinates.
r,z=YLCurve_beta(beta=0.2).curve()
plt.plot(r,z)
plt.axis('equal')
plt.show()

Qualitative relationship between beta and the droplet shape.

In [ ]:
betas=np.linspace(0.1,0.8,5)
plt.figure(figsize=(12,8))
for beta in betas:
    if beta==0.1:
        sr=3.3
    else:
        sr=4.0
    ylc=YLCurve_beta(beta=beta,srange=sr)
    plt.plot(np.hstack([-ylc.r[::-1],ylc.r]),np.hstack([ylc.z[::-1],ylc.z]),label=r'$\beta=%.2f$'%beta)
plt.axis('equal')
plt.grid()
plt.legend()
plt.savefig('betas.jpg')
plt.show()

In [ ]:
# Generate the dataset and save it as a pickle file.
beta_choices=np.linspace(0.2,0.7,100)
DataList=[]
for beta in beta_choices:
    DataList.append((YLCurve_beta(beta=beta).curve(),beta))
with open('./rawData.pkl','wb') as f:
    pickle.dump(DataList,f)

# 2. Neural Network Training

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader,sampler
import matplotlib.pyplot as plt
import pickle
import gc
from time import perf_counter
from sys import stdout
from layers import *
from utils import *
from matplotlib import rcParams

In [ ]:
#Load the simulated dataset before noise injection and resampling.
DataList=None
with open('./rawData.pkl','rb') as ds:
    DataList=pickle.load(ds)
# Restrict beta to approximately 0.3-0.7.
DataList=DataList[20::]

In [ ]:
print('Number of initial curves:', len(DataList))
beta = [i[1] for i in DataList]
print('Dataset beta range:', min(beta), ', ', max(beta))

In [ ]:
# Synthetic dataset with contour sampling.
# See the implementation below for the detailed sampling procedure.
class curveSetFull(Dataset):
    def __init__(self,raw,sample_size=100,device=torch.device('cuda')):
        super(curveSetFull,self).__init__()
        self.rawData=raw
        self.samples=None
        self.labels=None
        self.device=device
        self.sample_size=sample_size
        
    def sample(self,n,mode='uniform',pixel_quantize=False,noise_level=1e-3,sigma_offset=0.1):#n is the number of resampled examples per raw curve at different scales.
        if not self.samples is None:
            del self.samples
            del self.labels
            gc.collect()
            torch.cuda.empty_cache()
        
        self.samples=torch.zeros(len(self.rawData)*n,2,self.sample_size,
                                 requires_grad=False,dtype=torch.float32,device=self.device)
        self.labels=torch.zeros(2,len(self.rawData)*n,
                               requires_grad=False,dtype=torch.float32,device=self.device)
        for i in range(len(self.rawData)):
            sampleSize=np.int32(1000*np.random.uniform(0.8,1.0,(n,)))
            rawCurve,beta=self.rawData[i]
            for t,s in enumerate(sampleSize):
                if mode=='random':
                    num_sparse=torch.randint(5,10,(2,))
                    # Generate sparse sampling intervals.
                    # def fn(x,m=5,sigma=10):
                    #     x+=offset
                    #     return m*torch.exp(-x**2/sigma)
                    m=10
                    sigma2=10

                    sparsesl=[]
                    total_offsetl=0
                    for j in range(num_sparse[0].item()):
                        length=torch.randint(5,10,(1,)).item()
                        offset=-length//2
                        # o=torch.round(np.fromfunction(fn,(length,),m=10,sigma=15))+1
                        o=torch.round(m*torch.exp(((-torch.square(torch.arange(length)+offset))/sigma2)))+1
                        sparsesl.append(o)
                        total_offsetl+=torch.sum(o)

                    sparsesr=[]
                    total_offsetr=0
                    for j in range(num_sparse[1].item()):
                        length=torch.randint(5,10,(1,)).item()
                        offset=-length//2
                        # o=torch.round(np.fromfunction(fn,(length,),m=10,sigma=15))+1
                        o=torch.round(m*torch.exp(((-torch.square(torch.arange(length)+offset))/sigma2)))+1
                        sparsesr.append(o)
                        total_offsetr+=torch.sum(o)

                    original_distl=torch.linspace(0,s-total_offsetl-1,51,dtype=torch.int32).float()
                    diffl=torch.diff(original_distl)
                    #positions
                    positionsl=torch.randint(5,45,(num_sparse[0],)).squeeze()
                    for sparse, pos in zip(sparsesl,positionsl):
                        if pos+len(sparse)>len(diffl):
                            diffl[pos:]+=sparse[:len(diffl)-pos]
                        else:
                            diffl[pos:pos+len(sparse)]+=sparse
                    choices_left=torch.cumsum(diffl,0).int()

                    original_distr=torch.linspace(0,s-total_offsetr-1,51,dtype=torch.int32).float()
                    diffr=torch.diff(original_distr)
                    #positions
                    positionsr=torch.randint(5,45,(num_sparse[1],)).squeeze()
                    for sparse, pos in zip(sparsesr,positionsr):
                        if pos+len(sparse)>len(diffr):
                            diffr[pos:]+=sparse[:len(diffr)-pos]
                        else:
                            diffr[pos:pos+len(sparse)]+=sparse
                    choices_right=torch.cumsum(diffr,0).int()

                    # choices_left=np.sort(np.random.choice(np.arange(s),size=(50,),replace=False)) # Uniform random sampling introduces sampling noise.
                    # choices_right=np.sort(np.random.choice(np.arange(s),size=(50,),replace=False))
                if mode=='uniform':
                    choices_left=torch.linspace(0,s,50).int()
                    choices_right=choices_left
                else:
                    Exception('Mode not defined')
                
                rightCurve=rawCurve[::,choices_right]
                leftCurve=rawCurve[::,choices_left]
                leftCurve[0,::]*=-1
                leftCurve=leftCurve[:,::-1]
                self.samples[i*n+t]=torch.tensor(np.hstack((leftCurve,rightCurve)),dtype=torch.float32)
                self.labels[0,i*n+t]=torch.tensor(beta,dtype=torch.float32)
        
        self.samples=self.samples.transpose(1,2)
        
        # Randomly reflect the point coordinates.
        flip=torch.bernoulli(torch.ones(self.samples.shape[0],1,dtype=torch.float32)*0.5).cuda()*2-1
        self.samples[:,:,0]*=flip
        
        # Apply a small random rotation.
        theta=torch.randn(self.samples.shape[0]).cuda()*0.03
        rot_Matrix=torch.zeros(self.samples.shape[0],2,2).cuda()
        sint=torch.sin(theta)
        cost=torch.cos(theta)
        rot_Matrix[:,0,0]=cost
        rot_Matrix[:,0,1]=sint
        rot_Matrix[:,1,0]=-sint
        rot_Matrix[:,1,1]=cost
        self.samples=self.samples.matmul(rot_Matrix)
        
        
        # Add random noise.
        noise=torch.randn(*self.samples.shape,device=self.device,dtype=torch.float32)*noise_level
        self.samples+=noise
        
        # Apply a global offset.
        offsets=torch.randn(1,1,2,device=self.device,dtype=torch.float32)*sigma_offset
        self.samples+=offsets
        
        # Scale by a random R0.
        scale_factors=torch.randint(900,(self.samples.shape[0],1,1),
                                        dtype=torch.float32,device=self.device)+100
        self.samples*=scale_factors
        self.samples=self.samples.unsqueeze(1)
        if pixel_quantize:
            self.samples=torch.round(self.samples)
        self.labels[1,::]=scale_factors.squeeze()
    
    def gpu(self):
        self.samples=self.samples.cuda()
        self.labels=self.labels.cuda()
    
    def cpu(self):
        self.samples=self.samples.detach().cpu()
        self.labels=self.labels.detach.cpu()
        
    def __len__(self):
        if self.samples is None:
            Exception('Haven\'t sampled, the dataset is empty')
        else:
            return len(self.samples)
        
    def __getitem__(self,idx):
        return self.samples[idx],self.labels[::,idx]

In [ ]:
# Initialize the simulated dataset.
dsetFull=curveSetFull(DataList)

In [ ]:
# Resample the dataset and regenerate its noise.
# The first argument sets the repetitions per initial curve; 1 produces 1 * 80 = 80 examples.
dsetFull.sample(1,mode='random',noise_level=1e-3)

In [ ]:
# Visualize sampled examples from the synthetic dataset.
fig=plt.figure(figsize=(8,10))
for i in range(0, 4):
    plt.subplot(2, 2, i+1)
    r,z=dsetFull.samples[i*(len(dsetFull)//4)][0].transpose(0,1).detach().cpu()
    plt.scatter(r,z,s=2.0)
    plt.axis('equal')
    plt.xlabel(r'$r$')
    plt.ylabel(r'$z$')
    plt.title(r'$\beta$=%.2f'%dsetFull.labels[0][i*(len(dsetFull)//4)].item()+r', $R_0=%.1f$'%dsetFull.labels[1][i*(len(dsetFull)//4)].item())
fig.savefig('./SampledCurves.jpg')
plt.show()

In [ ]:
# Custom recurrent network for beta estimation.
class PendantGRU(nn.Module):
    def __init__(self,n_stack=4,h_dim=64,dropout=0.2):
        super(PendantGRU,self).__init__()
        self.rnn=nn.GRU(2,h_dim,n_stack,batch_first=True,dropout=dropout,bidirectional=True).cuda()
        self.linear=nn.Sequential(nn.Linear(2*h_dim,64),nn.ReLU(),
                                  nn.Linear(64,16),nn.ReLU(),
                                 nn.Linear(16,1)).cuda()
        self.ln=nn.LayerNorm((2*h_dim,)).cuda()
        
    def forward(self,x):
        out=self.rnn(x)[0][::,-1,::]
        out=self.ln(out)
        out=self.linear(out)
        return out
    
    def parametersRNN(self):
        for m in self.rnn.parameters():
            yield m
    
    def parametersLinear(self):
        for m in self.linear.parameters():
            yield m
            
    def parametersAttW(self):
        return self.crit
    

In [ ]:
# Convolutional network for R0 estimation.
class PendantInception(nn.Module):
    def __init__(self,in_channels,out_channels):
        super(PendantInception,self).__init__()
        if not out_channels%4==0:
            Exception('out_channels not supported')
        oc=out_channels//4
        self.conv3=nn.Conv2d(in_channels,oc,(3,1),padding='same',dtype=torch.float32)
        self.conv5=nn.Conv2d(in_channels,oc,(5,1),padding='same',dtype=torch.float32)
#         self.conv5=nn.Conv2d(in_channels,oc,(1,5),padding='same',dtype=torch.float32)
#         self.conv7=nn.Conv2d(in_channels,oc,(1,7),padding='same',dtype=torch.float32)
        
        self.conv32=nn.Conv2d(in_channels,oc*2,(3,2),padding=(1,0),dtype=torch.float32)
        self.conv52=nn.Conv2d(in_channels,oc*2,(5,2),padding=(2,0),dtype=torch.float32)
        
#        self.relu=nn.ReLU()
        
        self.modulelist=[self.conv3,self.conv5,
                         self.conv32,self.conv52,
                        # self.relu
                        ]

    def forward(self,x):
        out=[]
        for m in self.modulelist:
            out.append(m(x))
        y1=torch.cat(out[:2],dim=1)
        y2=torch.cat(out[2:],dim=3)
#         for y in [y1,y2,y3]:
#             print(y.shape)
        return torch.cat([y1,y2],dim=1)

R0=nn.Sequential(PendantInception(1,8),
                         PendantInception(8,16),
                         nn.MaxPool2d((2,1),stride=(2,1)),
                        PendantInception(16,16),
                         # PendantInception(16,16),
                        nn.MaxPool2d((2,1),stride=(2,1)),
                         PendantInception(16,16),
                         # PendantInception(16,16),
                         nn.MaxPool2d((2,1),stride=(2,1)),
                         nn.Conv2d(16,1,(12,2)),
                         nn.Flatten()).cuda()

In [ ]:
# Combine both networks to estimate the two parameters from one input.
class PendantNet(nn.Module):
    def __init__(self,beta,R0):
        super(PendantNet,self).__init__()
        self.beta=beta
        self.R0=R0
        
    def forward(self,x):
        pred_R0=self.R0(x).squeeze()
        #x_=x.squeeze()#/pred_R0.view(-1,1,1)
        x=x.squeeze(1)
        pred_beta=self.beta(x).squeeze()
        return pred_beta,pred_R0
            
    # def parametersOffset(self):
    #     for p in self.offset.parameters():
    #         yield p
    
    def train_(self,dataset,opt,scheduler=None,epochs=1,resample_every=5,batch_size=256,
          sample_parameters={'n':500,'mode':'random','pixel_quantize':True,'noise_level':1e-3}):
        self.train()
        switch=True
        for e in range(epochs):
            start=perf_counter()
            if ((e+1)%resample_every==0) or (e==0):
                gc.collect()
                torch.cuda.empty_cache()
                # if switch:
                #     sample_parameters['mode']='uniform'
                # else:
                #     sample_parameters['mode']='random'
                switch=not switch
                dataset.sample(**sample_parameters)
                N=len(dataset)
                loader=DataLoader(dataset,batch_size=batch_size,sampler=sampler.RandomSampler(range(len(dataset))))
            
            l1=0
            l2=0
            for x,y in loader:
                beta,R0=y.transpose(0,1)
                pred_beta,pred_R0=self.forward(x)
                loss2=F.mse_loss(pred_R0,R0,reduction='sum')
                loss1=F.mse_loss(pred_beta,beta,reduction='sum')                
                l1+=loss1.item()
                l2+=loss2.item()
                opt.zero_grad()
                loss2.backward(retain_graph=True)
                loss1.backward()
                opt.step()
            l1/=N
            l2/=N
            if not scheduler is None:
                scheduler.step()

            end=perf_counter()
            e_=e+1
            minu=int(end-start)//60
            sec=int(end-start)%60

            stdout.write('\r'+' '*100)
            stdout.write('\r epoch ({}/{})'.format(e_,epochs)+' took %d m %d s'%(minu,sec)+' loss = %.4f %.1f'%(l1,l2),)

In [ ]:
# Initialize the beta network; skip this section when using pretrained weights.
# Run the loading section below to restore previously trained models.
# The next three cells perform training and may be skipped when using pretrained weights.
pl=PendantGRU(n_stack=8,h_dim=128,dropout=0.1).cuda()

PN=PendantNet(pl,R0)

In [ ]:
DEFAULT_SAMPLE_PARAM={'n':500,'mode':'random','pixel_quantize':True,'noise_level':2e-3,'sigma_offset':0.1}
opt=optim.Adam([{'params':PN.beta.parametersRNN()},
               {'params':PN.beta.parametersLinear()},
               {'params':[torch.nn.Parameter(torch.tensor(PN.R0))], 'lr': 1e-3},
                #{'params':PN.R0.parametersLinear()}
               ],1e-4,weight_decay=1e-5)
sch=optim.lr_scheduler.StepLR(opt,step_size=20,gamma=0.99)

In [ ]:
PN.train_(dsetFull,opt,scheduler=sch,epochs=5000,sample_parameters=DEFAULT_SAMPLE_PARAM,resample_every=5)

In [ ]:
torch.save(pl.cpu(), './plCPU.pt')

In [ ]:
torch.save(R0.cpu(), './R0CPU.pt')

# 3. Experimental Inference

In [ ]:
# Load the models.
pl=torch.load('./plCPU.pt').cuda()
R0=torch.load('./R0CPU.pt').cuda()
PN=PendantNet(pl, R0)

In [ ]:
# Load the extracted contour.
arr = np.load('curve_data.npy')

In [ ]:
# Compute the needle diameter in pixels.
d_pixel=np.linalg.norm(arr[-1]-arr[0])
# Specify the physical needle diameter.
d=2.2
# Uniformly sample 100 points.
curve=sample(arr,100)
# Convert the contour to the network input format.
input_c=torch.tensor(curve,dtype=torch.float32).cuda().unsqueeze(0).unsqueeze(0)
# Set the model to evaluation mode.
PN.eval()
# Estimate the parameters with the model.
with torch.no_grad():
    beta,r0=PN(input_c)
    beta=beta.item()
    r0=r0.item()
    print(beta,r0)

In [ ]:
# Compute surface tension sigma.
R0=r0/d_pixel*d
sigma=9980*(R0*1e-3)**2/beta
print(d)
sigma

In [ ]:
# Compare the measured contour with the theoretical curve.
from scipy.integrate import odeint
class YLCurve_beta():
    def __init__(self,beta=0.2,iterations=1000,srange=4.0,r0=1e-6,theta0=1e-6):
        #Initialize variables.
        self.beta=beta
        self.s=np.linspace(0,srange,iterations)
        self.curves=[]
        
        #Define the differential system for scipy.odeint.
        def YLE(y,s,beta):
            x,z,theta=y
            dx=np.cos(theta)
            dz=np.sin(theta)
            dtheta=2-beta*z-np.sin(theta)/x
            return np.array([dx,dz,dtheta])
        #y1: x (horizontal coordinate)
        #y2: z (vertical coordinate)
        #y3: theta (tangent angle)
        #(s is the arc length measured from the bottom tip.)
        # Nondimensionalize each length by the radius of curvature at the bottom tip.
        
        #Solve the Young-Laplace equation.
        z0=r0*theta0
        solution,self.information=odeint(YLE,[r0,z0,theta0],self.s,args=(self.beta,),full_output=1)
        self.solution=solution[solution[::,2]<np.pi]
        self.r=solution[::,0]
        self.z=solution[::,1]

    def curve(self):
        return np.vstack([self.r,self.z])

In [ ]:
fig=plt.figure()
r,z=YLCurve_beta(beta=beta).curve()
plt.plot(r,z,color='blue',label='theoretical')
plt.scatter(curve[::,0]/r0,curve[::,1]/r0,s=2.0,label='detected')
plt.axis('equal')
plt.legend()
fig.savefig('./compare.jpg')
plt.show()